# 09. 배포용 데이터셋 만들기 + Hugging Face 업로드 — **GPU 필요 없음**

06이 만든 `dataset_main/`을 split별 폴더로 나누고, **test의 정답과 대본은 빼고** Hugging Face Datasets에 올린다.

| split | 올리는 파일 |
|---|---|
| train | `audio/`, `queries.json`, `gt.json`, `scripts.jsonl`, `meta.jsonl` |
| dev | `audio/`, `queries.json`, `gt.json`, `meta.jsonl` |
| test | `audio/`, `queries.json`, `meta.jsonl` (정답·대본 없음) |

test 정답은 Drive `dataset_main/private/`에만 둔다 (리더보드 채점용).

**준비**: Colab 보안 비밀에 `HF_TOKEN_WRITE` (Hugging Face Write 토큰) 등록

In [1]:
REPO_ID = "hyeonmin1/kbo-listen"  # Hugging Face 사용자이름/데이터셋이름
PRIVATE = False                     # True면 비공개로 올림 (제출 전에는 공개여야 채점자가 받을 수 있음)

import glob
import json
import os
import sys
from collections import Counter
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
_found = glob.glob("/content/drive/Othercomputers/*/kbolisten")
assert _found, "Drive '컴퓨터' 영역에서 kbolisten을 못 찾음 → Drive for desktop 동기화 확인"
sys.path.insert(0, os.path.dirname(_found[0]))

from kbolisten import build as B

DRIVE = Path("/content/drive/MyDrive/kbo-listen")
DATA = DRIVE / B.DIRS["main"]["dataset"]
REL = Path("/content/release")  # VM 디스크에 배포 폴더를 만든다 (음성은 복사 대신 링크 → Drive 용량을 더 안 씀)

meta = {}
with (DATA / "meta.jsonl").open(encoding="utf-8") as f:
    for line in f:
        m = json.loads(line)
        meta[m["clip_id"]] = m
queries = json.loads((DATA / "queries.json").read_text(encoding="utf-8"))
gt = json.loads((DATA / "gt.json").read_text(encoding="utf-8"))
scripts = {}
with (DATA / "scripts.jsonl").open(encoding="utf-8") as f:
    for line in f:
        s = json.loads(line)
        scripts[s["clip_id"]] = s
print(len(meta), "클립", Counter(m["split"] for m in meta.values()))

Mounted at /content/drive
1300 클립 Counter({'train': 1000, 'test': 200, 'dev': 100})


In [2]:
# 1) split별 배포 폴더
FILES = {"train": ("gt", "scripts"), "dev": ("gt",), "test": ()}  # queries·meta·audio는 모든 split


def dump(obj, path):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8")


def dump_lines(rows, path):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


for split, extra in FILES.items():
    ids = sorted(cid for cid, m in meta.items() if m["split"] == split)
    out = REL / split
    (out / "audio").mkdir(parents=True, exist_ok=True)
    for cid in ids:
        link = out / "audio" / f"{cid}.wav"
        if not link.exists():
            os.symlink(DATA / "audio" / f"{cid}.wav", link)
    dump({cid: queries[cid] for cid in ids}, out / "queries.json")
    dump_lines([meta[cid] for cid in ids], out / "meta.jsonl")
    if "gt" in extra:
        dump({cid: gt[cid] for cid in ids}, out / "gt.json")
    if "scripts" in extra:
        dump_lines([scripts[cid] for cid in ids], out / "scripts.jsonl")
    n_q = sum(len(queries[cid]) for cid in ids)
    print(f"{split:<5} {len(ids):>4}클립 질의 {n_q:>5}  파일 {sorted(p.name for p in out.iterdir())}")

# test 정답·대본은 Drive에만 (리더보드 채점용)
PRIV = DATA / "private"
PRIV.mkdir(exist_ok=True)
test_ids = sorted(cid for cid, m in meta.items() if m["split"] == "test")
dump({cid: gt[cid] for cid in test_ids}, PRIV / "test_gt.json")
dump_lines([scripts[cid] for cid in test_ids], PRIV / "test_scripts.jsonl")
print("비공개:", sorted(p.name for p in PRIV.iterdir()))

# 혹시 test 정답이 섞여 들어가지 않았는지 확인
assert not (REL / "test" / "gt.json").exists() and not (REL / "test" / "scripts.jsonl").exists()

train 1000클립 질의 16159  파일 ['audio', 'gt.json', 'meta.jsonl', 'queries.json', 'scripts.jsonl']
dev    100클립 질의  1534  파일 ['audio', 'gt.json', 'meta.jsonl', 'queries.json']
test   200클립 질의  3153  파일 ['audio', 'meta.jsonl', 'queries.json']
비공개: ['test_gt.json', 'test_scripts.jsonl']


In [3]:
# 2) 데이터셋 카드 (Hugging Face 첫 화면)
CARD = """---
license: cc-by-4.0
language:
- ko
task_categories:
- automatic-speech-recognition
pretty_name: KBO-Listen
size_categories:
- 1K<n<10K
---

# KBO-Listen: 야구 중계를 듣고 볼카운트 맞히기

야구 중계 음성을 듣고, 질의 시각마다 볼-스트라이크-아웃(B-S-O)을 맞히는 음성인식 챌린지 데이터셋입니다.
실제 KBO 경기 기록(2023~2026)을 바탕으로 합성한 중계 음성입니다 (가상의 캐스터·해설위원 목소리).

| split | 클립 (Easy / Hard) | 공개 파일 |
|---|---|---|
| train | 1,000 (500 / 500) | audio, queries, gt, scripts, meta |
| dev | 100 (50 / 50) | audio, queries, gt, meta |
| test | 200 (100 / 100) | audio, queries, meta (정답 비공개) |

- `audio/{clip_id}.wav`: 24kHz mono, 반 이닝 분량 (평균 약 97초, 총 약 35시간)
- `queries.json`: `{"clip_id": [t1, t2, ...]}` 질의 시각(초)
- `gt.json`: `{"clip_id": [{"t": 13.59, "b": 0, "s": 1, "o": 0}, ...]}`
- `scripts.jsonl`: 발화별 텍스트·화자·시작/끝 시각 (train만)
- `meta.jsonl`: 트랙(easy/hard), 태그, 시즌, 목소리 등

평가 코드·베이스라인·데이터 생성 과정: https://github.com/jhm991231/kbo-listen

경기 기록 출처: [slothman3878/kbo_playbyplay](https://huggingface.co/datasets/slothman3878/kbo_playbyplay) (CC BY 4.0).
음성 합성: Qwen3-TTS (Apache 2.0). Hard 트랙 대본 의역: Qwen3-32B (Apache 2.0).
"""
(REL / "README.md").write_text(CARD, encoding="utf-8")
print((REL / "README.md").read_text(encoding="utf-8")[:300])

---
license: cc-by-4.0
language:
- ko
task_categories:
- automatic-speech-recognition
pretty_name: KBO-Listen
size_categories:
- 1K<n<10K
---

# KBO-Listen: 야구 중계를 듣고 볼카운트 맞히기

야구 중계 음성을 듣고, 질의 시각마다 볼-스트라이크-아웃(B-S-O)을 맞히는 음성인식 챌린지 데이터셋입니다.
실제 KBO 경기 기록(2023~2026)을 바탕으로 합성한 중계 음성입니다 (가상의 캐스터·해설위원 목소리


In [4]:
# 3) 업로드. 끊겨도 다시 실행하면 이미 올린 파일은 건너뛴다 (upload_large_folder)
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_TOKEN_WRITE"))
print("로그인:", api.whoami()["name"])
api.create_repo(REPO_ID, repo_type="dataset", private=PRIVATE, exist_ok=True)
api.upload_large_folder(repo_id=REPO_ID, repo_type="dataset", folder_path=str(REL))
print("완료: https://huggingface.co/datasets/" + REPO_ID)

로그인: hyeonmin1


/tmp/ipykernel_803/7918533.py:7: FutureWarning: 
`upload_large_folder` is DEPRECATED and will be removed in a future release.

Use `upload_folder` instead:

    api.upload_folder(repo_id="hyeonmin1/kbo-listen", repo_type="dataset", folder_path="/content/release")

  api.upload_large_folder(repo_id=REPO_ID, repo_type="dataset", folder_path=str(REL))


Recovering from metadata files:   0%|          | 0/1310 [00:00<?, ?it/s]




---------- 2026-10-28 16:58:59 (0:00:00) ----------
Files:   hashed 0/1310 (0.0/6.1G) | pre-uploaded: 0/0 (0.0/6.1G) (+1310 unsure) | committed: 0/1310 (0.0/6.1G) | ignored: 0
Workers: hashing: 1 | get upload mode: 0 | pre-uploading: 0 | committing: 0 | waiting: 0
---------------------------------------------------

---------- 2026-10-28 16:59:59 (0:01:00) ----------
Files:   hashed 195/1310 (858.4M/6.1G) | pre-uploaded: 0/90 (0.0/6.1G) (+1210 unsure) | committed: 0/1310 (0.0/6.1G) | ignored: 0
Workers: hashing: 1 | get upload mode: 0 | pre-uploading: 0 | committing: 0 | waiting: 0
---------------------------------------------------
                             

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

  ...20260423HHLG02026_07T.wav:   1%|1         | 49.4kB / 3.56MB            

  ...20260619OBLG02026_08B.wav:   1%|1         | 48.8kB / 3.51MB            

  ...20260524SSLT02026_01T.wav:   1%|1         | 65.3kB / 4.70MB            

Validating                    :           |   0%            

완료: https://huggingface.co/datasets/hyeonmin1/kbo-listen
